In [ ]:
"""Assess agent's success based only on its execution trace using Claude Haiku"""

import anthropic
import json
import pandas as pd
import re
import time
import xml.etree.ElementTree as ET
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
from tqdm.notebook import tqdm

from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

client = anthropic.Anthropic()

SYSTEM_PROMPT = """
Your task is to analyze the execution trace of an LLM-powered AI agent and predict whether the agent succeeded in its task or not.
You will be provided with the task, the agent's trace (JSON format), and the agent's answer.
The agent can write and run Python code, perform web or wikipedia searches (provided via an API), and visit webpages.
The agent can only run for at most 10 steps, afterwards it will be forced to produce an answer.

Do not use your own knowledge to evaluate the agent's answer, just use the information in the trace.
Write your prediction in an XML tag like this:
<prediction>CHOOSE_EITHER_SUCCESS_OR_FAILURE</prediction>
""".strip()

USER_PROMPT = """
<task>
{task}
</task>

<agent_trace>
{trace}
</agent_trace>

<agent_answer>
{agent_output}
</agent_answer>

Now write your prediction here:
""".strip()

# MODEL_ID = "claude-sonnet-4-5-20250929"
MODEL_ID = "claude-3-5-haiku-20241022"


In [ ]:
def extract_full_trace(trace_path):
    with open(trace_path, "r") as f:
        raw_trace = json.load(f)

    # Hacky, used in eval
    def get_content(**kwargs):
        return kwargs.get("content")

    action_step = None
    for t in raw_trace[::-1]:
        if t["name"] == "ActionStep":
            action_step = t
            break

    if action_step is None:
        raise Exception("No ActionStep found")
    
    full_trace = []
    inputs = json.loads(action_step["attributes"]["output.value"])["model_input_messages"]
    output = json.loads(action_step["attributes"]["output.value"])["model_output_message"]

    for input_msg in inputs:
        content = eval(f"get_content({input_msg[input_msg.index('content='):]}")[0]["text"]
        role = re.search(r"MessageRole\.([A-Z_]+)", input_msg)
        role = role.group(1) if role is not None else ""
        full_trace.append({"role": role.lower(), "content": content})
    full_trace.append({"role": output["role"], "content": output["content"]})

    return full_trace

In [ ]:
model_size = "30"
input_path = f"../data/frames/profile_results_frames_full_llamacpp_qwen3_{model_size}b_judged_analyzed_predicted.csv"
df = pd.read_csv(input_path)
output_path = f"{input_path.split('.csv')[0]}_predicted.csv"
questions = df["question"]
answers = df["answer"]
agent_outputs = df["agent_output"]
agent_outputs_judgement = df["agent_output_eval"]

requests = []
for i in tqdm(range(len(questions))):
    trace = extract_full_trace(f"../logs/frames_full_llamacpp_qwen3_{model_size}b/{i}/run_0/raw/trace.json")
    prompt = USER_PROMPT.format(task=questions[i], trace=json.dumps(trace, indent=2), agent_output=agent_outputs[i])
    requests.append(Request(
        custom_id=f"{i}",
        params=MessageCreateParamsNonStreaming(
            model=MODEL_ID,
            max_tokens=1024,
            system=[{
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},
            }],
            messages=[{"role": "user", "content": prompt}],
        )
    ))

print(len(requests))

In [ ]:
# Submit batch
message_batch = client.messages.batches.create(requests=requests)
message_batch_id = message_batch.id
print(message_batch)

In [ ]:
while True:
    message_batch = client.messages.batches.retrieve(message_batch_id)
    if message_batch.processing_status == "ended":
        break
    time.sleep(10)
print("Ended!!!!!!!!!!!!!!!")

In [ ]:
# Save results
agent_output_prediction = [None] * len(questions)
for result in client.messages.batches.results(message_batch_id):
    custom_id = int(result.custom_id)
    match result.result.type:
        case "succeeded":
            res = result.result.message.content[0].text
            # match = re.search(r"<prediction>(.*?)</prediction>", res, re.DOTALL)
            # if match is None:
            #     print("Failed to find prediction in:\n" + res)
            #     continue
            # agent_output_prediction[custom_id] = match.group(1)
            agent_output_prediction[custom_id] = res
        case "errored":
            print(f"Request {custom_id} failed: {result.result.error.type}")
        case "expired":
            print(f"Request expired {custom_id}")

df["agent_output_prediction_haiku_3.5"] = agent_output_prediction

df.to_csv(output_path, index=False)

In [ ]:
preds = [None] * len(agent_output_prediction)
for i, pred in enumerate(agent_output_prediction):
    if pred is None:
        continue
    match = re.search(r"<prediction>(.*?)</prediction>", pred, re.DOTALL)
    if match is None:
        print("Failed to find prediction in:\n" + pred)
        continue
    t = match.group(1)
    if t in ["SUCCESS", "FAILURE"]:
        preds[i] = t
    else:
        preds[i] = "SUCCESS" if "SUCCESS" in t else "FAILURE"
df["agent_output_prediction_haiku_3.5_processed"] = preds

In [ ]:
df[df["agent_output_prediction_haiku_3.5_processed"] == "SUCCESS"]

pred = (df["agent_output_prediction_haiku_3.5_processed"] == "SUCCESS")   # predictions as booleans
true = (df["agent_output_eval"] == "CORRECT")   # ground truth as booleans

precision = precision_score(true, pred)
recall = recall_score(true, pred)
f1 = f1_score(true, pred)
accuracy = accuracy_score(true, pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Accuracy:", accuracy)